# 微调 MiniCPM5-2B 待办助手

此 Notebook 是 GPU 复现实验初稿，尚未在本次环境执行训练。请先阅读 Part 3 章节，使用支持 BF16 的 GPU 与 Python 3.11，上传完整项目。基座、模板、提示与测试集应固定。不会自动发布任何模型。

In [ ]:
from pathlib import Path
import sys, subprocess, json
candidates=[Path.cwd(), Path.cwd().parent, Path.cwd()/"minicpm-purplebook-draft"]
ROOT=next((p.resolve() for p in candidates if (p/"examples/task_contract.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("请先上传并解压完整实践包，再在项目目录打开Notebook")
sys.path.insert(0,str(ROOT/"examples"))
def run(args):
    subprocess.run([str(x) for x in args],cwd=ROOT,check=True)
print("Project:",ROOT)


## 1 安装官方指定模板版本
保留镜像的 CUDA / PyTorch 组合。安装后重启内核，并重新执行路径定位单元格。

In [ ]:
run([sys.executable,"-m","pip","install",
     "git+https://github.com/modelscope/ms-swift.git@654e24f17b5d9f40ed4b9ee4c56a723320244db5",
     "transformers==5.7.0","modelscope==1.33.0","accelerate","peft"])


In [ ]:
import torch
from importlib.metadata import version
assert torch.cuda.is_available(), "需要GPU Notebook"
assert torch.cuda.is_bf16_supported(), "本配置要求支持BF16的GPU，请更换镜像/资源或另行验证FP16路线"
(ROOT/"results").mkdir(exist_ok=True)
environment={name:version(name) for name in ["torch","transformers","ms-swift","modelscope","peft","accelerate"]}
environment["gpu"]=torch.cuda.get_device_name(0)
(ROOT/"results/environment.json").write_text(json.dumps(environment,indent=2),encoding="utf-8")
with (ROOT/"results/pip-freeze.txt").open("w") as f:
    subprocess.run([sys.executable,"-m","pip","freeze"],stdout=f,check=True)
print(environment)


## 2 下载魔搭基座并记录版本
第一次探索可以用 master。正式复现请从模型页复制实际固定版本/提交；完整文件哈希会写入清单。

In [ ]:
from modelscope import snapshot_download
import hashlib
revision=input("基座固定revision（首次探索可输入master）：").strip()
assert revision, "请填写实际revision"
base=Path(snapshot_download("OpenBMB/MiniCPM5-2B",revision=revision,
                          local_dir=str(ROOT/"models/minicpm5-2b")))
files=[]
for path in sorted(base.rglob("*")):
    if path.is_file() and not any(part.startswith(".") for part in path.relative_to(base).parts):
        with path.open("rb") as f: checksum=hashlib.file_digest(f,"sha256").hexdigest()
        files.append({"path":str(path.relative_to(base)),"bytes":path.stat().st_size,"sha256":checksum})
(ROOT/"results/base-manifest.json").write_text(json.dumps(
    {"model_id":"OpenBMB/MiniCPM5-2B","requested_revision":revision,"files":files},indent=2),encoding="utf-8")
print(base)


## 3 生成教学数据
训练 320、验证 64、测试 96 条，外围模板分离。这是合成教学集，不能代表真实用户分布。

In [ ]:
run([sys.executable,"examples/build_data.py"])
from task_contract import read_jsonl
print(read_jsonl(ROOT/"data/train.jsonl")[0])


## 4 测试基座
不使用测试结果调参；调参应使用 dev_gold.jsonl。这里保存第一次冻结协议下的测试结果。

In [ ]:
run([sys.executable,"examples/predict_todo.py","--model",base,"--out","results/base.jsonl"])
run([sys.executable,"examples/evaluate_todo.py","--predictions","results/base.jsonl",
     "--report","results/base-report.json"])


## 5 启动 LoRA 训练
下面是初稿资源配置。日志正常结束仍需通过任务评测。

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
run(["swift","sft","--model",base,"--model_type","llama","--template","minicpm5_2b",
     "--tuner_type","lora","--dataset","data/train.jsonl","--val_dataset","data/dev.jsonl",
     "--target_modules","q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj",
     "--lora_rank","16","--lora_alpha","32","--lora_dropout","0.05",
     "--num_train_epochs","2","--learning_rate","1e-4","--warmup_ratio","0.05",
     "--per_device_train_batch_size","1","--per_device_eval_batch_size","1",
     "--gradient_accumulation_steps","8","--max_length","512","--gradient_checkpointing","true",
     "--bf16","true","--eval_strategy","steps","--eval_steps","20",
     "--save_strategy","steps","--save_steps","20","--save_total_limit","2",
     "--logging_steps","5","--seed","47","--output_dir","runs/todo"])


## 6 选择实际检查点并合并
选择最后检查点可以复现流程；若要比较多个检查点，使用验证集选定后再冻结最终测试。

In [ ]:
checkpoints=sorted((ROOT/"runs/todo").rglob("checkpoint-*"))
checkpoints=[p for p in checkpoints if (p/"adapter_config.json").is_file()]
for i,p in enumerate(checkpoints): print(i,p)
if not checkpoints: raise RuntimeError("未找到适配器检查点，检查训练日志")
index=int(input("要导出的检查点编号："))
adapter=checkpoints[index]
merged=ROOT/"artifacts/todo-merged"
run(["swift","export","--model",base,"--model_type","llama","--template","minicpm5_2b",
     "--adapters",adapter,"--merge_lora","true","--output_dir",merged])
(ROOT/"results/training-artifacts.json").write_text(json.dumps(
    {"adapter":str(adapter),"merged":str(merged),"base_revision":revision},indent=2),encoding="utf-8")


## 7 测试合并模型
非法原始输出、超时和缺失预测均计失败，不修复 JSON 后重算。

In [ ]:
run([sys.executable,"examples/predict_todo.py","--model",merged,"--out","results/merged.jsonl"])
run([sys.executable,"examples/evaluate_todo.py","--predictions","results/merged.jsonl",
     "--report","results/merged-report.json"])
base_report=json.loads((ROOT/"results/base-report.json").read_text())
merged_report=json.loads((ROOT/"results/merged-report.json").read_text())
for key in ["schema_valid_rate","exact_record_rate","field_accuracy","null_preservation_rate"]:
    print(key,"base=",base_report[key],"merged=",merged_report[key])


## 8 转换 发布与手机验证

按 Part 3 第 6 至 9 节继续操作：用固定 llama.cpp 转换合并权重并量化；用同测试集评测 Q4 / Q8；为发布目录填写模型卡；创建自己的魔搭仓库后显式运行 upload_model.py；注册自定义模型并构建 Android App。

本 Notebook 不上传模型、不生成 APK，也不把 GPU 评测当成手机报告。模板见 `model-card-template.md`，手机实测表在章节中。